In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
from langchain_core.output_parsers import StrOutputParser

In [3]:
pdf_path = r"C:\Users\JayanthJella\Downloads\renewable_energy_guide.pdf"

loader = PyPDFLoader(pdf_path)
documents = loader.load()

print(f"Loaded {len(documents)} pages")


Loaded 5 pages


In [4]:
print("Number of documents:", len(documents))

print(documents[0].page_content[:1000])

Number of documents: 5
Renewable Energy: A Practical Guide
This document provides structured information about renewable energy, including its types, benefits,
challenges, applications, and common questions. It is designed as a sample knowledge document for
testing document-loading and Retrieval-Augmented Generation (RAG) systems.
1. Introduction
Renewable energy is energy obtained from natural sources that are replenished continuously or within a
relatively short period. Common sources include sunlight, wind, flowing water, organic matter, and heat
from the Earth. Renewable energy is increasingly used for electricity generation, heating, cooling,
transportation, and industrial applications.
2. Major Types of Renewable Energy
 Energy Source
How It Works
Common Application
Solar Energy
Solar panels convert sunlight into electricity; solar thermal systems use sunlight for heat.
Rooftop power, solar farms, water heating
Wind Energy
Wind turns turbine blades connected to a generator.
Onsho

In [5]:
text_splitter=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=50)
chunks=text_splitter.split_documents(documents)
print("Number of chunks:",len(chunks))

Number of chunks: 17


In [6]:
embeddings=OllamaEmbeddings(model="nomic-embed-text")

In [7]:
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

print("Vector database created.")

Vector database created.


In [8]:
retriever=vectorstore.as_retriever(search_kwargs={"k":6})

In [9]:
llm=ChatOllama(model="llama3.2",temperature=0)

In [10]:
prompt= ChatPromptTemplate.from_template("""
You are a document question-answering assistant.

Answer the question using ONLY the provided context.

If the answer is not available in the context, say:
"I could not find this information in the document."

Do not use outside knowledge.

At the end, mention the page number(s) used as the source.

Context:
{context}

Question:
{question}

Answer:
""")

In [11]:
def format_docs(docs):
    return "\n\n".join(f"[Page{doc.metadata.get('page',0)+1}]\n{doc.page_content}"
                       for doc in docs)

In [12]:
def ask_question(question):
    retrieved_docs=retriever.invoke(question)
    context= format_docs(retrieved_docs)

    messages=prompt.format_messages(context=context,question=question)
    response=llm.invoke(messages)

    print("\nAnswer:")
    print(response.content)

    print("\nSources:")

    pages=sorted(set(doc.metadata.get("page",0)+1
                     for doc in retrieved_docs))
    
    print ("Pages:",pages)

    #return response.content

In [13]:
ask_question("What are the major types of renewable engery")


Answer:
The major types of renewable energy are:

1. Solar Energy
2. Wind Energy
3. Hydropower
4. Biomass Energy
5. Geothermal Energy

Page 1 and 2.

Sources:
Pages: [1, 3, 4, 5]


In [14]:
ask_question("Why is energy storage useful?")


Answer:
Energy storage can hold electricity when supply is high and provide electricity later when demand is higher or renewable generation is lower.

Page 2.

Sources:
Pages: [1, 2, 4, 5]


In [15]:
ask_question("What is RAG? And Explain the difference between RAG and LLM?")


Answer:
I could not find this information in the document.

Page numbers used as the source: None (RAG is not explicitly defined in the provided context)

Sources:
Pages: [1, 2, 4, 5]
